# FedAvg on Non-IID Data

Compares FedAvg performance on IID, pathological shards, and Dirichlet label partitions.

In [1]:
import os
import pandas as pd
import torch
from src.utils import seed_everything, load_mnist, evaluate
from src.models import TwoNN
from src.fl import fedavg_round
from src.partition import load_partition

seed_everything(42)
os.makedirs('results', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(train_x, train_y), (test_x, test_y) = load_mnist('data')

partition_files = {
    'IID': 'data/iid_k100.npz',
    'Pathological': 'data/pathological_k100.npz',
    'Dirichlet 0.1': 'data/dirichlet_a01.npz',
    'Dirichlet 0.5': 'data/dirichlet_a05.npz'
}

Run FedAvg with E=5, B=10 across each partition.

In [2]:
records = []
rounds = 8
c_count = 10
e_val = 5
b_val = 10
lr = 0.03

for name, pfile in partition_files.items():
    part = load_partition(pfile)
    client_data = {c: (train_x[idx], train_y[idx]) for c, idx in part.items()}
    
    seed_everything(42)
    global_model = TwoNN().to(device)
    loss, acc = evaluate(global_model, test_x, test_y, device=device)
    records.append({'exp': '04_noniid', 'model': '2NN', 'algo': 'FedAvg', 'partition': name,
                    'K': 100, 'C': 0.1, 'E': e_val, 'B': b_val, 'lr': lr, 'seed': 42,
                    'round': 0, 'test_acc': acc, 'test_loss': loss})
    
    for rnd in range(1, rounds + 1):
        perm = torch.randperm(100)
        selected = perm[:c_count].tolist()
        fedavg_round(global_model, client_data, selected, epochs=e_val, batch_size=b_val, lr=lr, device=device)
        loss, acc = evaluate(global_model, test_x, test_y, device=device)
        records.append({'exp': '04_noniid', 'model': '2NN', 'algo': 'FedAvg', 'partition': name,
                        'K': 100, 'C': 0.1, 'E': e_val, 'B': b_val, 'lr': lr, 'seed': 42,
                        'round': rnd, 'test_acc': acc, 'test_loss': loss})
    print(f'{name} final accuracy: {acc:.4f}')

pd.DataFrame(records).to_csv('results/noniid_experiments.csv', index=False)

IID final accuracy: 0.9474


Pathological final accuracy: 0.6521


Dirichlet 0.1 final accuracy: 0.8276


Dirichlet 0.5 final accuracy: 0.9311


Non-IID partitioning degrades convergence speed and final accuracy, with pathological shards showing the largest penalty.